In [1]:
############################################################################################################################
# Libraries to import:
############################################################################################################################

import pandas as pd
import datetime
import webbrowser
import math
import folium
from haversine import haversine

############################################################################################################################
# Custom class which stores a pair of latitude and longitude coordinate points. This will initially
# store 2 random coordinates in a geographic area. Then we will loop through the optimization process,
# searching for better "central" points until it reaches the best pair of central points.
############################################################################################################################

class StateSpace:

    def __init__ (self, zipcode1, zipcode2, coordinate1, coordinate2, totalDistance):
        self.zipcode1 = zipcode1
        self.zipcode2 = zipcode2
        self.coordinate1 = coordinate1
        self.coordinate2 = coordinate2
        self.totaldistance = totalDistance

############################################################################################################################
# This function converts the zip code from float to integer:
############################################################################################################################

def ConvertZipCodeToInteger(row):
    val = 0
    if pd.notnull(row['Zipcode']):
        val = int(row['Zipcode'])
        
    return val

############################################################################################################################
# Main processing:
############################################################################################################################

# Import file with each zipcode and its latitiude/longitude into data frame. This will be used to plot each zipcode on a map.
print (datetime.datetime.now())
print ("\n Importing the zip code geocoding file...")
dfZipcodeGeocoded = pd.read_csv("zip_lat_long.csv", delimiter=',')

# Convert zipcode to integer, so we can join to another dataframe:
print ("\n Converting zipcodes to integers...")
dfZipcodeGeocoded['ZIP'] = dfZipcodeGeocoded['ZIP'].astype(int)

# Import a file of Nassau County Long Island zip codes:
print ("\n Importing the Nassau County zip code file...")
dfNassauCountyZipcodes = pd.read_csv("Zipcodes-NassauCountyLongIsland.csv")

# Convert zipcodes to integer, so we can join to another data frame:
print ("\n Converting zipcodes to integers...")
dfNassauCountyZipcodes['ZipCodeInteger'] = dfNassauCountyZipcodes.apply(ConvertZipCodeToInteger, axis=1)

# Merge both dataframes, so we can get the latitide/longtitude for every Nassau County zipcode:
print ("\n Geocoding zipcodes...")
dfNassauCountyZipcodes = dfNassauCountyZipcodes.merge(dfZipcodeGeocoded,left_on='ZipCodeInteger', right_on='ZIP')

# Let's randomly pick 2 coordinates to start. These will be our initial "central" points. But we will refine them during the optimization loops:
dfSample = dfNassauCountyZipcodes.sample (2)

# Load state space custom class with those random coordinates:
myStateSpace = StateSpace ( dfSample.iloc[0,1], dfSample.iloc[1,1], (dfSample.iloc[0,3], dfSample.iloc[0,4]), (dfSample.iloc[1,3], dfSample.iloc[1,4]), None)

# Calculate the total distance from those starting coordinates to the rest of the coordinates:
print ("\n Calculating total distance in State Space...")
myTotalDistance = 0

for index, row in dfNassauCountyZipcodes.iterrows():

    # This is the distance from coordinate #1 to the other points:
    distance1 = haversine ( (row["LAT"], row["LNG"]), (myStateSpace.coordinate1[0], myStateSpace.coordinate1[1]) )

    # This is the distance from coordinate #2 to the other points:
    distance2 = haversine ( (row["LAT"], row["LNG"]), (myStateSpace.coordinate2[0], myStateSpace.coordinate2[1]) )

    # Calculate the minimum of distance1 and distance2 and add it to the running total distance. The idea here is that in a pair of central
    # points, only the lower of both distances matter:
    myTotalDistance += min (distance1, distance2)

myStateSpace.totalDistance = myTotalDistance

# Now that the total distance have been calculted for both starting points, we can plot the starting coordinates on map. Then
# we can visually see where we started from.
StartingMap = folium.Map (location = [40.719678, -73.58386], zoom_start = 11)

# Add all Nassau County zipcode coordinates to Starting map in blue:
for index, row in dfNassauCountyZipcodes.iterrows():
    folium.Marker (location=[row["LAT"],row["LNG"]], popup=row["ZipCodeInteger"], icon=folium.Icon(color="blue")).add_to(StartingMap)

# Now we can add both initial starting points to Starting map in red:
folium.Marker (location=[myStateSpace.coordinate1[0],myStateSpace.coordinate1[1]], popup=myStateSpace.totalDistance, icon=folium.Icon(color="red")).add_to(StartingMap)
folium.Marker (location=[myStateSpace.coordinate2[0],myStateSpace.coordinate2[1]], popup=myStateSpace.totalDistance, icon=folium.Icon(color="red")).add_to(StartingMap)

# Save and display Starting map:
StartingMap.save("PythonOptimization-NassauCty-PairAlgorithm-StartingPoint.html")
webbrowser.open_new_tab ("PythonOptimization-NassauCty-PairAlgorithm-StartingPoint.html")

############################################################################################################################
# Optimization step:
############################################################################################################################

# Create a loop to randomly choose one coordinate and see if it is a better coordinate (more central than the current coordinates)
print ("\n Use loop to randomly pick coordinates and see if it is more central than the current coordinates...")
for loopcounter in range (0,2000):
    
    # Pick a random Nassau County zipcode coordinate:
    dfSample = dfNassauCountyZipcodes.sample(1)

    # Calculate total distance between the random point and the first current coordinate:
    myTotalDistance = 0
    for index, row in dfNassauCountyZipcodes.iterrows():
    
        distance1 = haversine ( (row["LAT"], row["LNG"]), (myStateSpace.coordinate1[0], myStateSpace.coordinate1[1]) )
        distance2 = haversine ( (row["LAT"], row["LNG"]), (dfSample.iloc[0,3], dfSample.iloc[0,4]) )
        myTotalDistance += min (distance1, distance2)

    # If the total distance of this random point & the current coordinate, then save that pair of coordinates:
    if myTotalDistance < myStateSpace.totalDistance:
        myStateSpace.zipcode2 = dfSample.iloc[0,1]
        myStateSpace.coordinate2 = (dfSample.iloc[0,3], dfSample.iloc[0,4])
        myStateSpace.totalDistance = myTotalDistance
       
    
    # Pick another random Nassau County zipcode coordinate:
    dfSample = dfNassauCountyZipcodes.sample(1)

    # This time around, calculate total distance between the random point and the 2nd current coordinate:
    myTotalDistance = 0
    for index, row in dfNassauCountyZipcodes.iterrows():
    
        distance1 = haversine ( (row["LAT"], row["LNG"]), (myStateSpace.coordinate2[0], myStateSpace.coordinate2[1]) )
        distance2 = haversine ( (row["LAT"], row["LNG"]), (dfSample.iloc[0,3], dfSample.iloc[0,4]) )
        myTotalDistance += min (distance1, distance2)

    # If the total distance of this random point & the current coordinate, then save that pair of coordinates:
    if myTotalDistance < myStateSpace.totalDistance:
        myStateSpace.zipcode1 = dfSample.iloc[0,1]
        myStateSpace.coordinate1 = (dfSample.iloc[0,3], dfSample.iloc[0,4])
        myStateSpace.totalDistance = myTotalDistance

# Optimization is done. No plot the final optimized coordinates on map:
EndingMap = folium.Map (location = [40.719678, -73.58386], zoom_start = 11)

# Add all Nassau County coordinates to map:
for index, row in dfNassauCountyZipcodes.iterrows():
    folium.Marker (location=[row["LAT"],row["LNG"]], popup=row["ZipCodeInteger"], icon=folium.Icon(color="blue")).add_to(EndingMap)

# Add final optimized coordinates to map:
folium.Marker (location=[myStateSpace.coordinate1[0],myStateSpace.coordinate1[1]], popup=myStateSpace.totalDistance, icon=folium.Icon(color="red")).add_to(EndingMap)
folium.Marker (location=[myStateSpace.coordinate2[0],myStateSpace.coordinate2[1]], popup=myStateSpace.totalDistance, icon=folium.Icon(color="red")).add_to(EndingMap)

# Save optimized map and open it in a browser to compare to the starting map:
EndingMap.save("PythonOptimization-NassauCty-PairAlgorithm-EndingPoint.html")
webbrowser.open_new_tab ("PythonOptimization-NassauCty-PairAlgorithm-EndingPoint.html")

print ("\n Completed")
print (datetime.datetime.now())


2025-06-03 19:23:12.294343

 Importing the zip code geocoding file...

 Converting zipcodes to integers...

 Importing the Nassau County zip code file...

 Converting zipcodes to integers...

 Geocoding zipcodes...

 Calculating total distance in State Space...

 Use loop to randomly pick coordinates and see if it is more central than the current coordinates...

 Completed
2025-06-03 19:23:24.666503
